# Глава 12. Мини-проект №2: «Предсказатель» 🎯

**После этого урока ты сможешь:**
- самостоятельно пройти полный цикл проекта: данные → Pipeline → оценка → вывод;
- честно оценить модель по кросс-валидации;
- сравнить свою модель с baseline и объяснить результат словами.

> ⚠️ **Про Colab.** Ноутбук состоит из двух частей.
> **Часть A** — рабочий пример, запускается сразу (кнопка ▶ сверху вниз).
> **Часть B** — пустой шаблон для ТВОЕГО датасета. В нём заглушки на кириллице
> (`"МОЙ_ДАТАСЕТ.csv"`, `"ЦЕЛЬ"`, `"ЧИСЛО1"`...). Если запустить его как есть — будет
> `FileNotFoundError`. Это нормально: сначала подставь свои данные и имена столбцов.

## Полный цикл проекта (5 шагов)

1. **Постановка задачи** — что предсказываем и зачем? Классификация или регрессия?
2. **Данные** — выбрать датасет, сделать EDA, почистить пропуски.
3. **Baseline** — простейшая модель как обязательная точка отсчёта.
4. **Модель в Pipeline** — собрать конвейер и честно оценить кросс-валидацией.
5. **Вывод** — сравнить с baseline, выбрать метрику, объяснить результат.

Это скелет любого ML-проекта — будем повторять его до автоматизма.

---
# ЧАСТЬ A. Рабочий пример (запускается сразу) ✅

Предсказываем выживаемость на «Титанике». Проходим все 5 шагов на реальных данных, чтобы увидеть, что каркас действительно работает.

In [ ]:
# Импорты — весь инструментарий фазы 2 в одном месте
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# --- Шаг 1-2. Данные: «Титаник» по ссылке (файл загружать не нужно) ---
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
X = df[["Age", "Fare", "Sex", "Pclass"]].copy()   # 3 числовых + 1 текстовый
y = df["Survived"]                                # 1 = выжил, 0 = нет
print("Размер данных:", X.shape)
X.head()

In [ ]:
# --- Шаг 3. Baseline: модель, которая всегда отвечает самым частым классом ---
# Оцениваем ТЕМ ЖЕ протоколом (кросс-валидация), что и основную модель — иначе сравнение нечестное
base = DummyClassifier(strategy="most_frequent")
base_score = cross_val_score(base, X, y, cv=5).mean()
print("Baseline (кросс-валидация):", round(base_score, 3))

In [ ]:
# --- Шаг 4. Модель в Pipeline ---
num = ["Age", "Fare", "Pclass"]   # числовые столбцы
cat = ["Sex"]                     # категориальные столбцы

prep = ColumnTransformer([
    # числа: заполнить медианой -> отмасштабировать
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num),
    # категории: заполнить самым частым -> закодировать 0/1
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat),
])

model = Pipeline([("prep", prep),
                  ("clf",  RandomForestClassifier(random_state=42))])

model_score = cross_val_score(model, X, y, cv=5).mean()
print("Модель (кросс-валидация):", round(model_score, 3))

In [ ]:
# --- Шаг 5. Вывод: сравниваем модель с baseline ---
print(f"Baseline: {base_score:.3f}")
print(f"Модель:   {model_score:.3f}")
diff = model_score - base_score
print(f"Модель обыгрывает baseline на {diff:+.3f}")
print("Вывод:", "модель полезна ✅" if diff > 0.02 else "модель не лучше простого угадывания ⚠️")

---
# ЧАСТЬ B. Шаблон для ТВОЕГО проекта 📝

Скопируй ячейку ниже, **замени заглавные заглушки** на свои данные и запусти.

Что менять:
- `"МОЙ_ДАТАСЕТ.csv"` → имя файла, который ты загрузил в Colab (значок папки слева → Загрузить),
  или ссылка на CSV;
- `"ЦЕЛЬ"` → столбец, который предсказываешь;
- `"ЧИСЛО1", "ЧИСЛО2"` → твои **числовые** столбцы;
- `"ТЕКСТ1"` → твои **категориальные** (текстовые) столбцы.

> Пока заглушки не заменены, ячейка выдаст `FileNotFoundError` — это ожидаемо.

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# 1. Данные — подставь свой файл и имена столбцов
df = pd.read_csv("МОЙ_ДАТАСЕТ.csv")
X = df.drop(columns=["ЦЕЛЬ"])
y = df["ЦЕЛЬ"]

# 2. Укажи свои столбцы
num = ["ЧИСЛО1", "ЧИСЛО2"]   # числовые
cat = ["ТЕКСТ1"]             # категориальные

# 3. Baseline (тем же протоколом, что и модель)
base = DummyClassifier(strategy="most_frequent")
print("Baseline (CV):", round(cross_val_score(base, X, y, cv=5).mean(), 3))

# 4. Pipeline
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat),
])
model = Pipeline([("prep", prep),
                  ("clf",  RandomForestClassifier(random_state=42))])

# 5. Честная оценка + вывод
print("Модель  (CV):", round(cross_val_score(model, X, y, cv=5).mean(), 3))

## Три слайда для защиты
Оформи результат как мини-презентацию: **задача → данные и результат → вывод.** Обязательно сравни модель с baseline.

## Как оценивается (макс. 13, успешно — от 8)
| Критерий | Баллы |
|---|---|
| Постановка задачи | 0–2 |
| Данные и Pipeline | 0–3 |
| ML и сравнение с baseline | 0–3 |
| Визуализация и метрики | 0–2 |
| Объяснение результата | 0–3 |

## Задания
**Базовый.** Пройди все 5 шагов на своём датасете и сравни модель с baseline.

**Со звёздочкой ⭐.** Сравни 2–3 модели (KNN, дерево, лес) по кросс-валидации и обоснуй выбор победителя.